# 02: RAG単体デモ

ChromaDB + GLM-4で医療QAを行う簡単なデモ。

## 1. ライブラリインポート

In [1]:
import os
import torch
from dotenv import load_dotenv
load_dotenv()

from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("インポート完了")

インポート完了


## 2. RAG初期化

In [2]:
# 設定
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 埋め込みモデル
embeddings = HuggingFaceEmbeddings(
    model_name=MODEL_NAME,
    model_kwargs={'device': device}
)

# ベクトルDB（Medical-RAGのDBを使用）
vectorstore = Chroma(
    persist_directory="../../02-Medical-RAG/results/chroma_db",
    embedding_function=embeddings
)

# GLM-4クライアント
llm = ChatOpenAI(
    model="GLM-4.7",
    base_url="https://api.z.ai/api/coding/paas/v4",
    api_key=os.getenv("OPENAI_API_KEY")
)

print(f"初期化完了: {device}")

C:\Users\hello\AppData\Local\Temp\ipykernel_11712\3041562949.py:12: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(


初期化完了: cuda


## 3. 質問応答

In [3]:
# 質問
question = "What are the side effects of insulin?"

# 検索
documents = vectorstore.similarity_search(question, k=3)

# コンテキスト作成
context = "\n\n".join([f"[{i}] {doc.page_content.strip()}" for i, doc in enumerate(documents, 1)])

# プロンプト
prompt = f"""Context: {context}
Question: {question}
Answer:"""

# 回答生成
response = llm.invoke(prompt)

print(f"Q: {question}\n")
print(f"A: {response.content}")

Q: What are the side effects of insulin?

A: The side effects of insulin can vary depending on the type of insulin used, the dosage, and the individual patient. Common side effects include:

*   **Hypoglycemia (Low Blood Sugar):** This is the most common side effect. Symptoms include shaking, sweating, rapid heartbeat, dizziness, hunger, confusion, and headache. Severe cases can lead to seizures or unconsciousness.
*   **Weight Gain:** Insulin can cause the body to store fat, leading to weight gain.
*   **Injection Site Reactions:** This includes redness, swelling, or itching at the spot where the insulin is injected. Lipodystrophy (thickening or pitting of the fat tissue) can also occur if the same injection site is used repeatedly.
*   **Fluid Retention (Edema):** Insulin may cause the body to retain salt and water, leading to swelling in the hands and feet.
*   **Hypokalemia (Low Potassium):** Insulin can cause potassium levels in the blood to drop.
*   **Allergic Reactions:** While

---

**次**: `03_integrated.ipynb` で統合パイプライン